# ViSEF — Priority 1: Multi-seed Benchmark (3 seed × 3 cấu hình)

Notebook này chạy lại **Ưu tiên 1** trong báo cáo:

1. **Dedup check** — phát hiện ảnh trùng Training↔Testing (aHash).
2. **9 lần chạy** = 3 seed × 3 cấu hình (`fedavg`, `fedprox`, `fedprox_dp`).
3. Mỗi lần: train → eval validation (mọi round) → chọn round tốt nhất theo val Macro-F1 → eval test **chỉ trên round đó**.
4. **Aggregate mean ± std** và vẽ error bars — đây là bằng chứng thống kê cần có cho hội đồng.

**Ước lượng thời gian** (T4): ~20 phút/run không DP, ~30 phút/run DP → tổng ~3.5 giờ. Có cơ chế resume: nếu cell bị ngắt, chạy lại sẽ bỏ qua run đã có `eval_test.json`.

**Trước khi chạy**: upload toàn bộ file `.py` (`common.py`, `client.py`, `server.py`, `partition.py`, `evaluate.py`, `privacy.py`, `run_simulation.py`, `prepare_data.sh`, `dedup_check.py`, `centralized.py`) vào `/content`.

In [ ]:
# 1) Cai dat moi truong
!pip install -q torch torchvision numpy Pillow matplotlib
!pip install -q opacus   # de --verify_dp doi chieu accountant
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2) Tai va chuan hoa du lieu
%cd /content
!sed -i 's/\r$//' prepare_data.sh
!bash prepare_data.sh

In [ ]:
# 3) Kiem tra cac file code da co
import os
required = ['common.py','client.py','server.py','partition.py','evaluate.py',
            'privacy.py','run_simulation.py','prepare_data.sh',
            'dedup_check.py','centralized.py']
missing = [f for f in required if not os.path.exists(f)]
assert not missing, f'Thieu file: {missing}'
print('OK - day du file:', required)

## Bước 1 — Dedup check (Priority 1, item 5)

Bộ Brain Tumor MRI Dataset (Kaggle) đã có báo cáo trùng lặp ảnh giữa Training và Testing. Nếu bỏ qua, accuracy có thể bị thổi phồng. Chạy kiểm tra trước; nếu phát hiện nhiều, chạy với `--remove` rồi **phân hoạch lại**.

In [ ]:
!python dedup_check.py --data_root ./brain-tumor-mri-dataset --threshold 0

# Neu co nhieu cap trung lap, bo comment dong duoi de xoa anh Training trung voi Testing:
# !python dedup_check.py --data_root ./brain-tumor-mri-dataset --threshold 0 --remove
# Sau khi --remove: phai phan hoach lai (REPARTITION=1 bash prepare_data.sh)

## Bước 2 — Cấu hình multi-seed

Đặt `QUICK_MODE=True` nếu muốn chạy thử nhanh (5 rounds, 1 seed). Bản nộp nên dùng cấu hình đầy đủ (3 seed × 20 rounds).

In [ ]:
QUICK_MODE = False   # bat True de chay thu nhanh

SEEDS   = [42, 123, 456] if not QUICK_MODE else [42]
ROUNDS  = 20 if not QUICK_MODE else 5
EPOCHS  = 3
BATCH   = 32
LR      = 1e-4
DOM_FRAC= 0.7
TRAIN_MODE = 'last_blocks'

CONFIGS = {
    'fedavg':      dict(mu=0.0,  dp=False),
    'fedprox':     dict(mu=0.01, dp=False),
    'fedprox_dp':  dict(mu=0.01, dp=True),
}
print(f'Seeds={SEEDS} | rounds={ROUNDS} | configs={list(CONFIGS)}')

## Bước 3 — Chạy 9 lần (có resume)

Mỗi lần: `run_simulation --skip_eval` → `evaluate --split val --all_rounds` → chọn round tốt nhất theo val Macro-F1 → `evaluate --split test --server_rounds <best>`.

Nếu cell bị ngắt, **chạy lại cell này** — run đã hoàn tất sẽ được bỏ qua.

In [ ]:
import subprocess, sys, json, time
from pathlib import Path

def sh(cmd):
    print('\n$', ' '.join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True)

def pick_best_round(run_dir):
    val = json.load(open(Path(run_dir) / 'eval' / 'eval_val.json'))['results']
    srv = [r for r in val if r['kind'] == 'server' and r['round'] > 0]
    if not srv:
        return None
    return max(srv, key=lambda r: r['macro_f1'])['round']

t_start = time.time()
for seed in SEEDS:
    for name, cfg in CONFIGS.items():
        tag = f'p1_{name}_seed{seed}'
        run_dir = Path('runs') / tag
        test_json = run_dir / 'eval' / 'eval_test.json'
        if test_json.exists():
            print(f'[SKIP] {tag} da hoan tat')
            continue
        print(f'\n========== {tag} ==========')
        cmd = [sys.executable, 'run_simulation.py',
               '--data_root', './brain-tumor-mri-dataset',
               '--tag', tag,
               '--partition_dir', 'partitions',
               '--rounds', str(ROUNDS), '--epochs', str(EPOCHS),
               '--batch_size', str(BATCH), '--lr', str(LR),
               '--train_mode', TRAIN_MODE,
               '--mu', str(cfg['mu']),
               '--dominant_frac', str(DOM_FRAC),
               '--seed', str(seed), '--skip_eval']
        if cfg['dp']:
            cmd += ['--dp', '--noise_multiplier', '1.0',
                    '--max_grad_norm', '1.0', '--verify_dp']
        if seed == SEEDS[0]:
            cmd.append('--repartition')
        sh(cmd)
        # validation moi round -> chon best round
        sh([sys.executable, 'evaluate.py', '--run_dir', run_dir,
            '--split', 'val', '--partition_dir', 'partitions',
            '--all_rounds', '--device', 'auto', '--no_plots'])
        best = pick_best_round(run_dir)
        print(f'[Pick] {tag}: best round = {best}')
        # test CHI tren best round
        sh([sys.executable, 'evaluate.py', '--run_dir', run_dir,
            '--split', 'test', '--data_root', './brain-tumor-mri-dataset',
            '--server_rounds', str(best), '--client_rounds', '',
            '--device', 'auto', '--no_plots'])

print(f'\nTong thoi gian: {(time.time()-t_start)/60:.1f} phut')

## Bước 4 — Aggregate mean ± std

Đọc `eval_test.json` của 9 run, tính mean và std trên 3 seed cho từng cấu hình. Đây là bảng sẽ đưa vào báo cáo.

In [ ]:
import pandas as pd, numpy as np

rows = []
for seed in SEEDS:
    for name in CONFIGS:
        tag = f'p1_{name}_seed{seed}'
        test_json = Path('runs') / tag / 'eval' / 'eval_test.json'
        if not test_json.exists():
            print(f'[MISS] {tag}'); continue
        t = json.load(open(test_json))['results'][0]
        val = json.load(open(Path('runs')/tag/'eval'/'eval_val.json'))['results']
        srv = [r for r in val if r['kind'] == 'server' and r['round'] > 0]
        best = max(srv, key=lambda r: r['macro_f1'])
        rows.append({
            'config': name, 'seed': seed, 'best_round': best['round'],
            'val_acc': best['accuracy'], 'val_f1': best['macro_f1'],
            'test_acc': t['accuracy'], 'test_f1': t['macro_f1'],
            'test_balacc': t['balanced_accuracy'],
            **{f'recall_{i}': r for i, r in enumerate(t['per_class_recall'])},
        })

df = pd.DataFrame(rows)
df.to_csv('runs/priority1_multiseed.csv', index=False)
df

In [ ]:
summary = df.groupby('config').agg({
    'test_acc':   ['mean','std'],
    'test_f1':    ['mean','std'],
    'test_balacc':['mean','std'],
}).round(4)
print('=== Test metrics: mean ± std (3 seeds) ===')
print(summary.to_string())
summary.to_csv('runs/priority1_summary.csv')

print('\n=== Recall theo lop (mean ± std) ===')
for i, cname in enumerate(['glioma','meningioma','notumor','pituitary']):
    sub = df.groupby('config')[f'recall_{i}'].agg(['mean','std']).round(3)
    print(f'\n{cname}:'); print(sub.to_string())

## Bước 5 — Biểu đồ error bars

Ba biểu đồ: accuracy, macro-F1, balanced accuracy — mean ± std trên 3 seed. Đây là hình chính cho báo cáo (thay cho bảng 1 lần chạy cũ).

In [ ]:
import matplotlib.pyplot as plt

configs = list(CONFIGS)
colors  = {'fedavg':'tab:blue','fedprox':'tab:orange','fedprox_dp':'tab:red'}
titles  = ['Accuracy','Macro-F1','Balanced Acc.']
cols    = ['test_acc','test_f1','test_balacc']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, col, title in zip(axes, cols, titles):
    means = [df[df['config']==c][col].mean()*100 for c in configs]
    stds  = [df[df['config']==c][col].std()*100  for c in configs]
    bars = ax.bar(configs, means, yerr=stds, capsize=8,
                  color=[colors[c] for c in configs], edgecolor='black')
    for b, m, s in zip(bars, means, stds):
        ax.text(b.get_x()+b.get_width()/2, m+s+1.0, f'{m:.1f}±{s:.1f}',
                ha='center', fontsize=9)
    ax.set_title(title); ax.set_ylabel('%'); ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('Priority 1 - Multi-seed benchmark (mean ± std, 3 seeds)', fontweight='bold')
fig.tight_layout()
fig.savefig('runs/priority1_summary.png', dpi=150)
plt.show()

In [ ]:
# Recall theo lop: so sanh 3 cau hinh
classes = ['glioma','meningioma','notumor','pituitary']
fig, ax = plt.subplots(figsize=(9,4.5))
x = np.arange(len(classes)); w = 0.25
for i, c in enumerate(configs):
    sub = df[df['config']==c]
    m = [sub[f'recall_{j}'].mean()*100 for j in range(4)]
    s = [sub[f'recall_{j}'].std()*100  for j in range(4)]
    ax.bar(x + i*w, m, w, yerr=s, capsize=3, label=c, color=colors[c])
ax.set_xticks(x + w); ax.set_xticklabels(classes)
ax.set_ylabel('Recall (%)'); ax.set_title('Per-class recall (mean ± std)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); fig.savefig('runs/priority1_recall.png', dpi=150)
plt.show()

## Bước 6 (tuỳ chọn) — Baseline centralized

Chạy centralized (60 epoch ≈ 20 vòng × 3 epoch, cùng ngân sách update) để trả lời câu hỏi *"FL mất bao nhiêu so với huấn luyện tập trung?"*. **Không bắt buộc** cho Priority 1 nhưng rất nên có trong báo cáo.

In [ ]:
for seed in SEEDS:
    tag = f'p1_centralized_seed{seed}'
    run_dir = Path('runs') / tag
    if (run_dir/'global'/'global_round_00.pt').exists():
        print(f'[SKIP] {tag}'); continue
    sh([sys.executable, 'centralized.py',
        '--data_root', './brain-tumor-mri-dataset',
        '--run_dir', run_dir,
        '--epochs', str(ROUNDS*EPOCHS),
        '--batch_size', str(BATCH), '--lr', str(LR),
        '--train_mode', TRAIN_MODE, '--seed', str(seed)])
    sh([sys.executable, 'evaluate.py', '--run_dir', run_dir,
        '--split', 'test', '--data_root', './brain-tumor-mri-dataset',
        '--server_rounds', '0', '--client_rounds', '',
        '--device', 'auto', '--no_plots'])

## Bước 7 — Xuất file cho báo cáo

Tải xuống `runs/priority1_summary.csv`, `runs/priority1_multiseed.csv`, `runs/priority1_summary.png`, `runs/priority1_recall.png` để chèn vào LaTeX.

In [ ]:
from google.colab import files
for f in ['runs/priority1_summary.csv','runs/priority1_multiseed.csv',
          'runs/priority1_summary.png','runs/priority1_recall.png']:
    if Path(f).exists():
        files.download(f)